# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. The dataset describes clinicopathological variables of cancer survivors with second primary colorectal cancer, including anatomical, molecular, and comorbidity information.

### Dataset Source
The dataset is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the FAIR^2 dataset's metadata and records using `mlcroissant`. The metadata includes description, author, and relevant dataset properties.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Access metadata (do not treat it as a dict; use its attributes)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"Published: {meta.datePublished}")
print(f"Keywords: {meta.keywords}")
print(f"Data Collection Timeframe: {meta.dataCollectionTimeframe}")

## 2. Data Overview
Review available record sets, fields, their `@id`s (Croissant identifiers), and summaries. By convention, all entities are referenced by their `@id` for reproducibility.

_Note: Record sets may contain fields that are further mapped to columns. The record sets defined in FAIR^2 are found in the metadata property._

In [ ]:
# Show record sets and their field IDs

record_set_ids = []
if meta.recordSet:
    print("Available record sets:")
    for rs in meta.recordSet:
        print(f"  - RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")
        record_set_ids.append(rs['@id'])

# For each record set, display field @id and name
for rs_id in record_set_ids:
    rs = next(r for r in meta.recordSet if r['@id'] == rs_id)
    print(f"\nFields in RecordSet '{rs.get('name', 'N/A')}' (@id={rs_id}):")
    if 'field' in rs:
        for f in rs['field']:
            print(f"    Field @id: {f['@id']}, name: {f.get('name', 'N/A')}, dataType: {f.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All extraction uses the record set and field `@id` as defined in the previous section.

_Below, all record set `@id`s are used as a reference according to metadata._

In [ ]:
# Prepare to load all record sets into dataframes

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded RecordSet '{rs_id}' with columns: {df.columns.tolist()}")

# Display the head of the first record set
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"\nExample records from RecordSet '@id={primary_rs_id}':")
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Now, let's filter, normalize, and group data from a numeric field using its `@id`, as specified in the schema.

_Example: Suppose the record set contains a numeric field such as 'Age'. We'll use the field's `@id` for all processing._

In [ ]:
# Identify numeric fields (e.g. Age) and a grouping field (e.g. Sex or MSI status)

# Let's programmatically find a numeric field, e.g. 'Age'.
# We'll use the first record set and attempt to locate 'Age'.
primary_rs = next(r for r in meta.recordSet if r['@id'] == primary_rs_id)
numeric_field_id = None
group_field_id = None
for f in primary_rs.get('field', []):
    if f.get('dataType') in ['Integer', 'Float']:
        if 'age' in f.get('name', '').lower():
            numeric_field_id = f['@id']
    if 'sex' in f.get('name', '').lower() or 'msi' in f.get('name', '').lower():
        group_field_id = f['@id']

# For this demonstration, if not found, fall back to column name
df = dataframes[primary_rs_id]

if numeric_field_id and numeric_field_id in df.columns:
    field_id = numeric_field_id
else:
    # Default to a column likely named 'Age', otherwise use the first numeric column
    field_id = None
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        if 'age' in col.lower():
            field_id = col
            break
    if not field_id and len(numeric_cols) > 0:
        field_id = numeric_cols[0]
numeric_field_id = field_id

if group_field_id and group_field_id in df.columns:
    group_field = group_field_id
else:
    group_field = None
    for c in df.columns:
        if 'sex' in c.lower() or 'msi' in c.lower():
            group_field = c
            break

# Filter records with Age > threshold
threshold = 50
if numeric_field_id:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with '{numeric_field_id}' > {threshold}:")
    display(filtered_df.head())

    # Normalize Age
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field (e.g. Sex or MSI status)
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
else:
    print("No numeric field found for analysis.")

## 5. Visualization
Visualize the data distribution for the selected numeric field and explore relationships between fields.

_Example: Show histogram of Age, and a boxplot of Age by MSI status (or Sex)._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of '{numeric_field_id}' in RecordSet '{primary_rs_id}'")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"'{numeric_field_id}' grouped by '{group_field}'")
        plt.show()

## 6. Conclusion

We have explored the FAIR^2 Croissant dataset describing clinicopathological features of second primary colorectal cancer in cancer survivors. Key steps included:
* Loading descriptive metadata and record sets by their Croissant `@id`
* Extracting tabular records for analysis
* Processing numeric variables (e.g., Age) and visualizing distributions
* Normalizing and grouping data using field identifiers

This workflow provides a reproducible foundation for more advanced clinical and molecular hypothesis testing using FAIR data standards.